# Monitor Training with ClickHouse

This notebook demonstrates how to query ClickHouse for real-time training
metrics and visualize training progress.

## Prerequisites
- ClickHouse accessible (configured via `CLICKHOUSE_HOST` env var)
- Training jobs running or completed (Phase 05 + 07)

In [ ]:
import os
import clickhouse_connect
import pandas as pd
import matplotlib.pyplot as plt

# Connect to ClickHouse
ch_client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "clickhouse.logging.svc.cluster.local"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    database=os.environ.get("CLICKHOUSE_DB", "isaac_lab"),
)
print(f"Connected to ClickHouse: {ch_client.server_version}")

## Query Recent Training Jobs

List all training jobs from the last 24 hours.

In [ ]:
# Query recent training jobs
query = """
SELECT
    job_id,
    job_name,
    environment,
    algorithm,
    status,
    started_at,
    duration_seconds
FROM training_jobs
WHERE started_at >= now() - INTERVAL 24 HOUR
ORDER BY started_at DESC
LIMIT 20
"""

df_jobs = ch_client.query_df(query)
df_jobs

In [ ]:
# Query training metrics for a specific job
job_id = df_jobs.iloc[0]["job_id"] if len(df_jobs) > 0 else "example-job-id"

metrics_query = f"""
SELECT
    iteration,
    reward_mean,
    reward_std,
    policy_loss,
    value_loss,
    entropy,
    timestamp
FROM training_metrics
WHERE job_id = '{job_id}'
ORDER BY iteration ASC
"""

df_metrics = ch_client.query_df(metrics_query)
print(f"Retrieved {len(df_metrics)} metric records for job {job_id}")
df_metrics.head(10)

## Visualize Training Progress

In [ ]:
if len(df_metrics) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Training Progress: {job_id}", fontsize=14)

    # Mean reward
    axes[0, 0].plot(df_metrics["iteration"], df_metrics["reward_mean"], color="blue")
    axes[0, 0].fill_between(
        df_metrics["iteration"],
        df_metrics["reward_mean"] - df_metrics["reward_std"],
        df_metrics["reward_mean"] + df_metrics["reward_std"],
        alpha=0.2,
    )
    axes[0, 0].set_title("Mean Reward")
    axes[0, 0].set_xlabel("Iteration")

    # Policy loss
    axes[0, 1].plot(df_metrics["iteration"], df_metrics["policy_loss"], color="red")
    axes[0, 1].set_title("Policy Loss")
    axes[0, 1].set_xlabel("Iteration")

    # Value loss
    axes[1, 0].plot(df_metrics["iteration"], df_metrics["value_loss"], color="green")
    axes[1, 0].set_title("Value Loss")
    axes[1, 0].set_xlabel("Iteration")

    # Entropy
    axes[1, 1].plot(df_metrics["iteration"], df_metrics["entropy"], color="purple")
    axes[1, 1].set_title("Entropy")
    axes[1, 1].set_xlabel("Iteration")

    plt.tight_layout()
    plt.show()
else:
    print("No metrics data available to plot.")

In [ ]:
# Query GPU utilization for running jobs
gpu_query = """
SELECT
    job_id,
    avg(gpu_utilization) AS avg_gpu_util,
    max(gpu_memory_used_mb) AS peak_gpu_mem_mb,
    avg(gpu_temperature_c) AS avg_gpu_temp
FROM gpu_metrics
WHERE timestamp >= now() - INTERVAL 1 HOUR
GROUP BY job_id
ORDER BY avg_gpu_util DESC
"""

df_gpu = ch_client.query_df(gpu_query)
print("GPU utilization by job:")
df_gpu